
# Keras + TensorBoard: Scalars and Image Grid per Epoch

This notebook trains a simple CNN on MNIST, logs **loss/accuracy** to TensorBoard every batch/epoch, and logs an **image grid with predictions** after each epoch.


In [ ]:

%pip -q install --upgrade pip tensorboard tensorflow>=2.11


In [ ]:

import os, time, math
import numpy as np
import tensorflow as tf

# Reproducibility
tf.random.set_seed(42)
np.random.seed(42)

print(tf.__version__)


In [ ]:

# Load MNIST
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

# Split validation
x_val, y_val = x_test[:5000], y_test[:5000]
x_test, y_test = x_test[5000:], y_test[5000:]

# Scale to [0,1] and add channel dimension
x_train = (x_train.astype("float32")/255.0)[..., None]
x_val   = (x_val.astype("float32")/255.0)[..., None]
x_test  = (x_test.astype("float32")/255.0)[..., None]

x_train.shape, x_val.shape, x_test.shape


In [ ]:

from tensorflow.keras import layers as L, models as M

def make_model():
    inputs = L.Input(shape=(28,28,1))
    x = L.Conv2D(32, 3, activation="relu")(inputs)
    x = L.MaxPooling2D()(x)
    x = L.Conv2D(64, 3, activation="relu")(x)
    x = L.MaxPooling2D()(x)
    x = L.Flatten()(x)
    x = L.Dense(128, activation="relu")(x)
    outputs = L.Dense(10, activation="softmax")(x)
    return M.Model(inputs, outputs)

model = make_model()
model.summary()


In [ ]:

class ImageGridCallback(tf.keras.callbacks.Callback):
    def __init__(self, x_val, y_val, class_names=None, tag="val", max_images=16, log_dir="logs/exp"):
        super().__init__()
        self.x = x_val
        self.y = y_val
        self.class_names = class_names
        self.tag = tag
        self.max_images = max_images
        self.writer = tf.summary.create_file_writer(log_dir)
        # fix a slice to keep grids comparable
        self.idx = np.arange(min(max_images, x_val.shape[0]))

    def _make_grid(self, imgs, cols=8):
        # imgs: (N,H,W,C) float in [0,1]
        n = imgs.shape[0]
        cols = min(cols, n)
        rows = int(math.ceil(n/cols))
        H, W, C = imgs.shape[1:]
        grid = np.ones((rows*H, cols*W, C), dtype=imgs.dtype)
        for i in range(n):
            r, c = divmod(i, cols)
            grid[r*H:(r+1)*H, c*W:(c+1)*W, :] = imgs[i]
        return grid

    def on_epoch_end(self, epoch, logs=None):
        x = self.x[self.idx]
        y_true = self.y[self.idx]

        y_prob = self.model.predict(x, verbose=0)
        y_pred = np.argmax(y_prob, axis=1)

        imgs = x.copy()
        if imgs.ndim == 3:      # (N,H,W)
            imgs = imgs[..., None]
        if imgs.shape[-1] == 1: # expand to 3 channels for TB
            imgs = np.repeat(imgs, 3, axis=-1)

        grid = self._make_grid(imgs, cols=8)
        lines = []
        for i in range(imgs.shape[0]):
            t = self.class_names[y_true[i]] if self.class_names is not None else str(y_true[i])
            p = self.class_names[y_pred[i]] if self.class_names is not None else str(y_pred[i])
            lines.append(f"{i:02d}: true={t} pred={p}")

        with self.writer.as_default():
            # log scalars if available on logs
            if logs is not None:
                if "val_loss" in logs:
                    tf.summary.scalar("loss/val_epoch", logs["val_loss"], step=epoch)
                if "val_accuracy" in logs:
                    tf.summary.scalar("acc/val_epoch", logs["val_accuracy"], step=epoch)
            tf.summary.image(f"{self.tag}/grid", np.expand_dims(grid, 0), step=epoch)
            tf.summary.text(f"{self.tag}/labels", "\n".join(lines), step=epoch)
        self.writer.flush()


In [ ]:

stamp = time.strftime("%Y%m%d-%H%M%S")
log_dir = f"logs/keras-mnist-{stamp}"

tb = tf.keras.callbacks.TensorBoard(
    log_dir=log_dir,
    update_freq='batch',   # batch-level scalar logging
    histogram_freq=0,
    write_graph=True,
    write_images=False)

img_cb = ImageGridCallback(x_val, y_val, class_names=None, tag="val", max_images=16, log_dir=log_dir)

E = 5
B = 128

model.compile(optimizer="adam",
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])

history = model.fit(
    x_train, y_train,
    validation_data=(x_val, y_val),
    epochs=E,
    batch_size=B,
    callbacks=[tb, img_cb],
)


In [ ]:

model.evaluate(x_test, y_test, verbose=2)


In [ ]:

%load_ext tensorboard
%tensorboard --logdir logs



## Notes
- The `TensorBoard` callback logs batch-level `loss` and `accuracy`. The custom `ImageGridCallback` logs a prediction grid and label text each epoch.
- Keep the same validation slice across epochs to make grids comparable.
- For grayscale inputs, the callback repeats the channel to 3 for TensorBoard image support.
- Change `E`, `B`, and `make_model()` as needed.
